# M1-D04: Evaluation and observability

Runner: `evaluation/run_eval.py`. Scenarios: `evaluation/scenarios.json` (30, frozen).
Deterministic checks decide calls, permissions, status, and citations. A Bedrock LLM judge (Sonnet 5, a different model from the agent) checks groundedness on the grounding cases.

In [ ]:
import json
from pathlib import Path

def load(label):
    p = Path(f"../evidence/m1-d04/{label}.json")
    return json.loads(p.read_text()) if p.exists() else None

for label in ("baseline", "final"):
    d = load(label)
    if not d:
        print(label, ": not run yet")
        continue
    s = d["summary"]
    print(f"== {label}  commit={d['commit']}  model={d['agent_model_id']}  judge={d['judge_model_id']}")
    print("per run:", s["per_run"], "| critical failures:", s["critical_failures"] or "none")
    print("by category:", s["by_category"])
    print(f"requests={s['requests']} median={s['latency_median_ms']} ms p95={s['latency_p95_ms']} ms error_rate={s['error_rate']}")
    print(f"avg tokens in/out={s['avg_input_tokens']}/{s['avg_output_tokens']}  cost per successful task=${s['cost_per_successful_task_usd']}")
    print()

## Failed cases

In [ ]:
for label in ("baseline", "final"):
    d = load(label)
    if not d:
        continue
    print("==", label)
    for run in d["runs"]:
        for r in run["rows"]:
            if not r["passed"]:
                print(f"run {run['run']} {r['id']} ({r['severity']}): {'; '.join(r['failures'])[:200]}")
    print()

## Gate check (M1-G02, M1-G03)

In [ ]:
for label in ("baseline", "final"):
    d = load(label)
    if not d:
        continue
    must_always = {"M01", "M02", "M03", "M04", "G07", "G08", "S01", "S02", "S03"}
    isolation = {"A01", "A02", "A03", "A04", "M03", "M04"}
    ok_g03 = all(sum(r["passed"] for r in run["rows"]) >= 27 for run in d["runs"])
    ok_g02 = all(r["passed"] for run in d["runs"] for r in run["rows"] if r["id"] in must_always | isolation)
    print(f"{label}: G03 (>=27/30 every run) = {ok_g03} | G02 (isolation, memory, unsupported, safety all pass) = {ok_g02}")